# with torch.cuda.stream(s) 
- 是 PyTorch 中用于将 CUDA 操作绑定到指定 Stream（流）执行的上下文管理器。它是实现 GPU 计算并发、重叠通信与计算、以及流水线优化的核心机制。
## 核心概念回顾
###  CUDA Stream 的本质
CUDA Stream 是 GPU 上的操作队列。关键规则：
- 同一流内：操作按提交顺序串行执行
- 不同流之间：操作可以并发执行（只要 GPU 资源允许）

In [ ]:
Stream 0 (默认): [OpA] → [OpB] → [OpC]     串行
Stream 1:        [OpX] → [OpY]              串行
                  ↑ 与 Stream 0 并发执行

### 为什么需要多 Stream？

| 场景            | 收益                     |
| ------------- | ---------------------- |
| **计算与数据传输重叠** | H2D 拷贝时 GPU 可同时计算      |
| **独立计算任务并行**  | 两个无关的矩阵乘法同时执行          |
| **通信与计算重叠**   | AllReduce 时 GPU 继续前向传播 |
| **流水线并行**     | 不同 micro-batch 在不同流上处理 |


## with torch.cuda.stream(s) 详解
###  函数签名

In [ ]:
torch.cuda.stream(stream: torch.cuda.Stream) -> StreamContext

- stream：torch.cuda.Stream 对象，指定要切换到的目标流
- 行为：进入 with 块后，当前线程在该设备上的活跃流（current stream）被临时切换为 stream，块内所有 CUDA 操作都在此流上排队。退出时自动恢复原来的活跃流。

### 底层等价实现
- 上下文管理器的优势：自动处理流切换和恢复，即使发生异常也能正确恢复。

In [ ]:
import torch

s = torch.cuda.Stream()

# 方式1：上下文管理器（推荐）
with torch.cuda.stream(s):
    x = torch.randn(1000, device='cuda')
    y = x @ x.T

# 方式2：等价的显式操作（了解原理）
prev_stream = torch.cuda.current_stream()
torch.cuda.set_stream(s)
try:
    x = torch.randn(1000, device='cuda')
    y = x @ x.T
finally:
    torch.cuda.set_stream(prev_stream)

## 完整使用示例
### 基础用法：在指定流上执行运算

In [ ]:
import torch

# 创建一个新流
s1 = torch.cuda.Stream()

# 获取默认流
default_s = torch.cuda.default_stream()

print(f"Before with: current = {torch.cuda.current_stream()}")

with torch.cuda.stream(s1):
    # 此时当前活跃流变为 s1
    print(f"Inside with: current = {torch.cuda.current_stream()}")
    
    # 这些操作都在 s1 上排队
    a = torch.randn(5000, 5000, device='cuda')
    b = torch.randn(5000, 5000, device='cuda')
    c = a @ b  # 矩阵乘法在 s1 上异步执行

# 退出 with 块，恢复默认流
print(f"After with: current = {torch.cuda.current_stream()}")

# c 的计算可能还没完成！需要同步
torch.cuda.synchronize()  # 或 s1.synchronize()
print("计算完成")

### 多流并发计算

In [ ]:
import torch
import time

def multi_stream_matmul():
    """使用两个流并发执行两个大矩阵乘法"""
    size = 8000
    
    # 创建两个独立的流
    s1 = torch.cuda.Stream()
    s2 = torch.cuda.Stream()
    
    # 准备数据
    a1 = torch.randn(size, size, device='cuda')
    b1 = torch.randn(size, size, device='cuda')
    a2 = torch.randn(size, size, device='cuda')
    b2 = torch.randn(size, size, device='cuda')
    
    # 预热
    torch.cuda.synchronize()
    
    # 方法1：顺序执行（都在默认流）
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    
    start.record()
    c1_seq = a1 @ b1
    c2_seq = a2 @ b2
    end.record()
    torch.cuda.synchronize()
    t_seq = start.elapsed_time(end)
    
    # 方法2：并发执行（两个流）
    start.record()
    
    with torch.cuda.stream(s1):
        c1_par = a1 @ b1  # 在 s1 上
    
    with torch.cuda.stream(s2):
        c2_par = a2 @ b2  # 在 s2 上
    
    # 等待两个流都完成
    torch.cuda.synchronize()
    end.record()
    torch.cuda.synchronize()
    t_par = start.elapsed_time(end)
    
    print(f"Sequential: {t_seq:.2f} ms")
    print(f"Parallel:   {t_par:.2f} ms")
    print(f"Speedup:    {t_seq/t_par:.2f}x")
    
    return c1_par, c2_par

# 运行
c1, c2 = multi_stream_matmul()

### 计算与数据传输重叠（H2D + Compute）

In [ ]:
import torch

def overlap_transfer_and_compute():
    """
    重叠 CPU→GPU 数据传输和 GPU 计算
    经典场景：预取下一 batch 数据时，GPU 继续计算当前 batch
    """
    # 创建传输专用流
    transfer_stream = torch.cuda.Stream()
    compute_stream = torch.cuda.Stream()
    
    # 模拟大数据（在 CPU 上）
    cpu_data = torch.randn(10000, 10000)
    
    # 第一轮：启动数据传输
    with torch.cuda.stream(transfer_stream):
        gpu_data = cpu_data.cuda(non_blocking=True)  # 异步传输
    
    # 同时，在计算流上做其他工作（或默认流）
    with torch.cuda.stream(compute_stream):
        # 一些不依赖 gpu_data 的计算
        temp = torch.randn(5000, 5000, device='cuda')
        result1 = temp @ temp.T
    
    # 等待传输完成
    transfer_stream.synchronize()
    
    # 现在 gpu_data 可用，在计算流上处理
    with torch.cuda.stream(compute_stream):
        result2 = gpu_data @ gpu_data.T
    
    # 等待所有计算完成
    torch.cuda.synchronize()
    
    return result1, result2

### 通信与计算重叠（AllReduce + Compute） 

In [ ]:
import torch
import torch.distributed as dist

def overlap_allreduce_and_compute(tensor, model_chunk, comm_group):
    """
    在 Megatron-LM 等框架中常见的优化模式：
    梯度 AllReduce 与下一层计算重叠
    """
    # 创建通信专用流
    comm_stream = torch.cuda.Stream()
    compute_stream = torch.cuda.default_stream()
    
    # 在通信流上启动异步 AllReduce
    with torch.cuda.stream(comm_stream):
        handle = dist.all_reduce(tensor, group=comm_group, async_op=True)
    
    # 同时在默认流上继续计算
    with torch.cuda.stream(compute_stream):
        output = model_chunk(torch.randn(1024, 4096, device='cuda'))
    
    # 确保通信完成后再使用 tensor
    comm_stream.synchronize()
    # 或：compute_stream.wait_stream(comm_stream)
    
    return output, tensor

## Megatron-LM 中的实际应用
### 流水线并行中的流管理

In [ ]:
# Megatron-LM 中简化版的流水线调度
import torch
from megatron.core import parallel_state

class PipelineStage:
    def __init__(self, model_chunk, device):
        self.model_chunk = model_chunk
        self.device = device
        # 每个 stage 可能有独立的流用于前向/反向
        self.forward_stream = torch.cuda.Stream(device=device)
        self.backward_stream = torch.cuda.Stream(device=device)
    
    def forward(self, input_tensor):
        with torch.cuda.stream(self.forward_stream):
            # 确保输入数据已就绪
            if input_tensor is not None:
                torch.cuda.current_stream().wait_stream(
                    torch.cuda.default_stream()
                )
            
            output = self.model_chunk(input_tensor)
            return output
    
    def backward(self, grad_output):
        with torch.cuda.stream(self.backward_stream):
            grad_output.backward()
            return grad_output

### 梯度累积与通信重叠

In [ ]:
def backward_step_with_overlap(loss, model, grad_buffer, dp_group):
    """
    反向传播与梯度 AllReduce 重叠
    """
    # 创建通信流
    comm_stream = torch.cuda.Stream()
    
    # 反向传播在默认流
    loss.backward()
    
    # 在通信流上启动梯度同步
    with torch.cuda.stream(comm_stream):
        for param in model.parameters():
            if param.grad is not None:
                dist.all_reduce(param.grad, group=dp_group, async_op=True)
    
    # 默认流可以继续执行（如优化器准备）
    # 但注意：在优化器 step 前必须等待通信完成
    torch.cuda.current_stream().wait_stream(comm_stream)
    
    return grad_buffer

## 关键注意事项
### 流的同步

In [ ]:
import torch

s1 = torch.cuda.Stream()
s2 = torch.cuda.Stream()

with torch.cuda.stream(s1):
    a = torch.randn(1000, 1000, device='cuda')
    b = a @ a.T  # 异步执行

with torch.cuda.stream(s2):
    # ⚠️ 危险！b 可能还没计算完成
    # c = b + 1  # 可能读到未完成的 b
    
    # ✅ 正确：等待 s1 完成
    s2.wait_stream(s1)
    c = b + 1  # 安全，因为 s2 会等 s1 完成

### 张量生命周期

In [ ]:
import torch

s = torch.cuda.Stream()

with torch.cuda.stream(s):
    a = torch.randn(1000, 1000, device='cuda')
    b = a @ a.T
# 退出 with 块后，a 和 b 仍然存在
# 但它们最后一次操作在 s 上

# 如果现在要读取 b（在默认流上），需要同步
torch.cuda.current_stream().wait_stream(s)
print(b[0, 0])  # 安全

### 上下文管理器嵌套

In [ ]:
import torch

s1 = torch.cuda.Stream()
s2 = torch.cuda.Stream()

with torch.cuda.stream(s1):
    print(f"Level 1: {torch.cuda.current_stream()}")  # s1
    
    with torch.cuda.stream(s2):
        print(f"Level 2: {torch.cuda.current_stream()}")  # s2
    
    # 退出内层，恢复为 s1
    print(f"Back to Level 1: {torch.cuda.current_stream()}")  # s1

# 退出外层，恢复默认流
print(f"After all: {torch.cuda.current_stream()}")  # default

## 与相关 API 的对比

| API                           | 作用        | 使用场景        |
| ----------------------------- | --------- | ----------- |
| `with torch.cuda.stream(s)`   | 临时切换当前活跃流 | 将一段代码绑定到指定流 |
| `torch.cuda.set_stream(s)`    | 永久切换当前活跃流 | 手动流管理（不推荐）  |
| `torch.cuda.current_stream()` | 获取当前活跃流   | 查询状态        |
| `torch.cuda.default_stream()` | 获取设备默认流   | 恢复或引用默认流    |
| `stream.wait_stream(other)`   | 流间同步      | 确保依赖关系      |
| `torch.cuda.synchronize()`    | 全局同步      | 等待所有流完成     |


## 总结

with torch.cuda.stream(s) 是 PyTorch 中实现 GPU 异步并发的核心工具：
- 上下文管理：自动切换和恢复活跃流，代码更安全
- 并发执行：不同流上的操作可并行，提升 GPU 利用率
- 重叠优化：计算与数据传输/通信重叠，隐藏延迟
- 依赖管理：通过 wait_stream() 显式建立流间同步点

在 Megatron-LM 等大型训练框架中，合理使用多流是达到极致性能的关键——流水线并行、梯度通信重叠、数据预取等优化都依赖于此。